# 1. Khai báo thư viện (Imports)
Nạp các module phục vụ cho toàn bộ quy trình đánh giá:
* **Xử lý Tensor & Mô hình:** `torch`, `torch.nn`, `torchvision.transforms` để chuẩn hóa ảnh, nội suy và đưa dữ liệu lên GPU.
* **Mô hình OCR & Đồ họa:** `PaddleOCR`, `PIL.Image` để đọc, cắt và tiền xử lý ảnh đầu vào.
* **Đo lường & Đánh giá:** `editdistance` để tính toán Normalized Edit Distance (NED).
* **Quản lý dữ liệu & Trực quan:** `pandas`, `tqdm.notebook`, `matplotlib.pyplot` để theo dõi tiến trình và xuất bảng số liệu.

In [5]:
import glob
import re
import numpy as np
import torch.nn as nn
from PIL import Image
import pandas as pd
import os

import editdistance
from IPython.display import display

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from torchvision import datasets, models
from torchvision import transforms as T
from torch.optim.lr_scheduler import CosineAnnealingLR
from __future__ import print_function
import torch.nn.functional as F
import cv2

# 2. Tải mô hình OCR & Định nghĩa các hàm phụ trợ
* Khởi tạo kiến trúc OCR: **PARSeq** (từ PyTorch Hub).
* `parse_totaltext_annotation(txt_path)`: Trích xuất tọa độ polygon/bounding box (`xmin`, `ymin`, `xmax`, `ymax`) và lọc nhãn không hợp lệ (`#`, `###`, chuỗi rỗng) từ tập Total-Text.
* `load_perturbation(path)` & `apply_perturbation(...)`: Tải và áp dụng ma trận nhiễu UAP ($224 \times 224$) lên ảnh tensor với giới hạn độ nhiễu $\epsilon$.
* `predict_parseq` & `predict_paddle`: Giải mã kết quả dự đoán chuỗi ký tự từ tensor đầu vào.
* `compute_ned`: Tính khoảng cách chỉnh sửa chuẩn hóa giữa nhãn Ground Truth và kết quả OCR.
* `compute_cer`: Character Error Rate — tỉ lệ lỗi ở cấp ký tự, tính bằng khoảng cách chỉnh sửa giữa pred và gt chia cho độ dài ground-truth (0 nếu gt rỗng và pred cũng rỗng, ngược lại phạt tối đa = 1.0).
* `compute_wer`: Word Error Rate — tương tự CER nhưng tính khoảng cách chỉnh sửa ở cấp từ (tách chuỗi bằng khoảng trắng trước khi so sánh), phản ánh mức độ sai lệch ở tầm câu/cụm từ thay vì từng ký tự đơn lẻ.

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

parseq = torch.hub.load('baudm/parseq', 'parseq', pretrained=True).eval().to(device)

def load_perturbation(path):
    uap = torch.load(path, map_location=device)
    if isinstance(uap, dict) and 'perturbation' in uap:
        uap = uap['perturbation']
    if uap.dim() == 3:
        uap = uap.unsqueeze(0)
    if uap.shape[-2:] != (224, 224):
        uap = F.interpolate(uap, size=(224, 224), mode='bilinear', align_corners=False)
    return uap.to(device)

def apply_perturbation(img_tensor, uap, eps=16/255.0):
    uap_clamped = torch.clamp(uap, -eps, eps)
    return torch.clamp(img_tensor + uap_clamped, 0.0, 1.0)

uap_base_dir = "/kaggle/input/datasets/tinphan2007/ocr-perturbations"

model_files = {
    "ConvNeXt-Tiny": "vision_perturbation/uap_convnext_tiny.pt",
    "DenseNet-121":  "vision_perturbation/uap_densenet121.pt",
    "ResNet-50":     "vision_perturbation/uap_resnet50.pt",
    "VGG-16":        "vision_perturbation/uap_vgg16.pt",
    "ViT-Base":      "vision_perturbation/uap_vit_base.pt",
    "CLIP":          "vlm_perturbation/uap_clip.pt",
    "EVA-CLIP":      "vlm_perturbation/uap_eva_clip.pt",
    "MetaCLIP":      "vlm_perturbation/uap_metaclip.pt",
    "OpenCLIP":      "vlm_perturbation/uap_open_clip.pt",
    "SigLIP":        "vlm_perturbation/uap_siglip.pt",
}

model_category = {
    "ConvNeXt-Tiny": "Vision", "DenseNet-121": "Vision", "ResNet-50": "Vision",
    "VGG-16": "Vision", "ViT-Base": "Vision",
    "CLIP": "VLM", "EVA-CLIP": "VLM", "MetaCLIP": "VLM", "OpenCLIP": "VLM", "SigLIP": "VLM",
}

uaps = {}
for attack_type in ["Targeted", "Untargeted"]:
    for model_name, rel_path in model_files.items():
        full_path = os.path.join(uap_base_dir, attack_type, rel_path)
        if os.path.exists(full_path):
            uaps[(model_name, attack_type)] = load_perturbation(full_path)
        else:
            print(f"Cảnh báo: Không tìm thấy {full_path}")

print(f"Đã nạp thành công {len(uaps)}/20 UAP tensors (Targeted + Untargeted).")
assert len(uaps) > 0, "Không nạp được UAP nào — kiểm tra lại đường dẫn!"

parseq_norm = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def predict_parseq(model, img_tensor):
    norm_tensor = parseq_norm(img_tensor.clone())
    with torch.no_grad():
        logits = model(norm_tensor)
        pred = logits.softmax(-1)
        label, _ = model.tokenizer.decode(pred)
        return label[0].lower().strip()

def compute_ned(pred, gt):
    if len(gt) == 0 and len(pred) == 0:
        return 1.0
    dist = editdistance.eval(pred, gt)
    return 1.0 - (dist / max(len(pred), len(gt)))

def compute_cer(pred, gt):
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    dist = editdistance.eval(pred, gt)
    return dist / len(gt)

def compute_wer(pred, gt):
    pred_words = pred.split()
    gt_words = gt.split()
    if len(gt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    dist = editdistance.eval(pred_words, gt_words)
    return dist / len(gt_words)

def parse_totaltext_annotation(txt_path):
    crops_info = []
    if not os.path.exists(txt_path):
        return crops_info

    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            trans_match = re.search(r"transcriptions:\s*\[\s*u?['\"](.*?)['\"]\s*\]", line, re.IGNORECASE)
            if trans_match:
                text = trans_match.group(1).strip()
            else:
                matches = re.findall(r"['\"](.*?)['\"]", line)
                text = matches[-1].strip() if matches else ""

            x_match = re.search(r"x:\s*\[\[?([0-9\s,]+)\]?\]", line)
            y_match = re.search(r"y:\s*\[\[?([0-9\s,]+)\]?\]", line)

            if x_match and y_match:
                x_pts = [int(p) for p in re.findall(r'\d+', x_match.group(1))]
                y_pts = [int(p) for p in re.findall(r'\d+', y_match.group(1))]
            else:
                parts = line.split(',')
                if len(parts) >= 9:
                    pts = [int(p) for p in re.findall(r'\d+', ','.join(parts[:8]))]
                    x_pts = pts[0::2]
                    y_pts = pts[1::2]
                    text = parts[8].strip().strip("'").strip('"')
                else:
                    continue

            if text in ['#', '###', ''] or len(x_pts) == 0 or len(y_pts) == 0:
                continue

            xmin, xmax = max(0, min(x_pts)), max(x_pts)
            ymin, ymax = max(0, min(y_pts)), max(y_pts)

            if xmax > xmin and ymax > ymin:
                crops_info.append(((xmin, ymin, xmax, ymax), text.lower()))

    return crops_info

Using cache found in /root/.cache/torch/hub/baudm_parseq_main


Đã nạp thành công 20/20 UAP tensors (Targeted + Untargeted).


# 3. Pipeline Đánh giá Zero-shot Robustness trên Total-Text
Thực hiện đánh giá khả năng chống chịu tấn công đối kháng trên tập Test (Total-Text):
1. **Đồng bộ khung ảnh:** Đưa ảnh gốc về kích thước chuẩn $(1, 3, 224, 224)$.
2. **Áp dụng UAP:** Tạo ảnh đối kháng bằng cách cộng trực tiếp `uap_vgg` và `uap_vlm` lên toàn khung ảnh $224 \times 224$.
3. **Ánh xạ tọa độ & Cắt vùng chữ:** Co giãn bounding box sang tỷ lệ $224 \times 224$, sau đó cắt vùng chữ từ cả ảnh sạch và 2 ảnh đối kháng để bảo toàn ngữ cảnh nhiễu.
4. **Chuẩn hóa đầu vào OCR:** Resize từng crop về $(32, 128)$ và đưa vào PARSeq/PaddleOCR.
5. **Ghi nhận số liệu:** Thống kê độ chính xác (Accuracy), NED, CER, WER và tỷ lệ tấn công thành công (ASR).

In [7]:
img_dir = "/kaggle/input/datasets/ipythonx/totaltextstr/Total-Text/Test"
annot_dir = "/kaggle/input/datasets/ipythonx/totaltextstr/Total-Text/Annotation/groundtruth_polygonal_annotation/Test"
img_files = sorted(glob.glob(os.path.join(img_dir, "*.*")))

resize_224 = T.Resize((224, 224))
resize_ocr = T.Resize((32, 128))
to_tensor = T.ToTensor()

uap_keys = list(uaps.keys()) 

metrics = {
    key: {'correct': 0, 'ned': 0.0, 'cer': 0.0, 'wer': 0.0}
    for key in uap_keys
}
total_samples = 0 

for img_p in tqdm(img_files, desc="Đang đánh giá 20 UAP (Targeted + Untargeted)"):
    base_name = os.path.splitext(os.path.basename(img_p))[0]
    annot_candidates = [
        os.path.join(annot_dir, f"poly_gt_{base_name}.txt"),
        os.path.join(annot_dir, f"{base_name}.txt"),
        os.path.join(annot_dir, f"gt_{base_name}.txt")
    ]
    txt_p = next((p for p in annot_candidates if os.path.exists(p)), None)
    if not txt_p:
        continue

    crops = parse_totaltext_annotation(txt_p)
    if not crops:
        continue

    try:
        full_img = Image.open(img_p).convert('RGB')
        orig_w, orig_h = full_img.size

        img_224 = to_tensor(resize_224(full_img)).unsqueeze(0).to(device)

        pert_imgs = {key: apply_perturbation(img_224, uap_tensor) for key, uap_tensor in uaps.items()}

        scale_x, scale_y = 224.0 / orig_w, 224.0 / orig_h

        for (xmin, ymin, xmax, ymax), gt in crops:
            x1 = int(np.clip(xmin * scale_x, 0, 223))
            y1 = int(np.clip(ymin * scale_y, 0, 223))
            x2 = int(np.clip(xmax * scale_x, 1, 224))
            y2 = int(np.clip(ymax * scale_y, 1, 224))

            if (x2 - x1) < 2 or (y2 - y1) < 2:
                continue

            clean_crop = resize_ocr(img_224[:, :, y1:y2, x1:x2])
            pred_clean = predict_parseq(parseq, clean_crop)
            is_clean_correct = (pred_clean == gt)

            if not is_clean_correct:
                continue

            total_samples += 1

            for key in uap_keys:
                adv_crop = resize_ocr(pert_imgs[key][:, :, y1:y2, x1:x2])
                pred_adv = predict_parseq(parseq, adv_crop)

                is_adv_correct = (pred_adv == gt)
                if is_adv_correct:
                    metrics[key]['correct'] += 1
                metrics[key]['ned'] += compute_ned(pred_adv, gt)
                metrics[key]['cer'] += compute_cer(pred_adv, gt)
                metrics[key]['wer'] += compute_wer(pred_adv, gt)

    except Exception as e:
        continue

print(f"\nTổng số text instance clean-correct dùng để đánh giá: {total_samples}")

Đang đánh giá 20 UAP (Targeted + Untargeted):   0%|          | 0/300 [00:00<?, ?it/s]


Tổng số text instance clean-correct dùng để đánh giá: 1593


# 4. Tổng hợp kết quả & hiển thị bảng so sánh
* `Clean Acc`: luôn = 100% theo thiết kế (chỉ giữ lại các mẫu PARSeq đọc đúng trên ảnh gốc).
* `Adv Acc`: % mẫu vẫn được đọc đúng sau khi bị áp nhiễu UAP.
* `ASR` (Attack Success Rate): tỉ lệ PARSeq bị đánh lừa.
* `Adv NED` / `Adv CER` / `Adv WER`: đo mức độ sai lệch chi tiết ở cấp ký tự và cấp từ giữa dự đoán trên ảnh nhiễu và ground-truth.

In [9]:
rows = []
for (model_name, attack_type), d in metrics.items():
    adv_acc = d['correct'] / total_samples * 100
    clean_acc = 100.0
    asr= clean_acc - adv_acc
    adv_ned = (d['ned'] / total_samples) * 100
    adv_cer = d['cer'] / total_samples
    adv_wer = d['wer'] / total_samples

    rows.append({
        'Category': model_category[model_name],
        'Source Model': model_name,
        'Attack Type': attack_type,
        'Clean Acc': clean_acc,
        'Adv Acc': adv_acc,
        'Adv NED': adv_ned,
        'Adv CER': adv_cer,
        'Adv WER': adv_wer,
        'ASR': asr,
    })

df = pd.DataFrame(rows)
df = df.sort_values(by=['Attack Type', 'ASR'], ascending=[True, False]).reset_index(drop=True)

def style_table(df_sub, title=""):
    styled = (
        df_sub.style
        .format({
            'Clean Acc': '{:.2f}%',
            'Adv Acc': '{:.2f}%',
            'Adv NED': '{:.2f}%',
            'Adv CER': '{:.4f}',
            'Adv WER': '{:.4f}',
            'ASR': '{:.2f}%',
        })
        .background_gradient(subset=['Adv Acc'], cmap='Blues')
        .background_gradient(subset=['ASR'], cmap='Reds')
        .background_gradient(subset=['Adv NED'], cmap='Blues_r')
        .set_caption(title)
        .set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]}])
    )
    return styled

print("\n=== BẢNG TỔNG HỢP - TARGETED ATTACK (nhưng đo untargeted trên OCR) ===")
df_targeted = df[df['Attack Type'] == 'Targeted'].drop(columns=['Attack Type']).reset_index(drop=True)
display(style_table(df_targeted, "Targeted-sourced Perturbations → Untargeted Attack on OCR"))

print("\n=== BẢNG TỔNG HỢP - UNTARGETED ATTACK ===")
df_untargeted = df[df['Attack Type'] == 'Untargeted'].drop(columns=['Attack Type']).reset_index(drop=True)
display(style_table(df_untargeted, "Untargeted-sourced Perturbations → Untargeted Attack on OCR"))

print("\n=== BẢNG GỘP CẢ 20 PERTURBATION ===")
display(style_table(df, "So sánh Targeted vs Untargeted Perturbations trên PARSeq OCR"))

df.to_csv("/kaggle/working/ocr_uap_comparison_20.csv", index=False)
print("\nĐã lưu bảng đầy đủ tại: /kaggle/working/ocr_uap_comparison_20.csv")


=== BẢNG TỔNG HỢP - TARGETED ATTACK (nhưng đo untargeted trên OCR) ===


,Category,Source Model,Clean Acc,Adv Acc,Adv NED,Adv CER,Adv WER,ASR
0,Vision,ViT-Base,100.00%,76.21%,85.86%,0.1495,0.2379,23.79%
1,VLM,OpenCLIP,100.00%,76.77%,86.22%,0.1434,0.2323,23.23%
2,VLM,MetaCLIP,100.00%,76.84%,86.06%,0.1469,0.2316,23.16%
3,VLM,EVA-CLIP,100.00%,77.21%,86.77%,0.1415,0.2279,22.79%
4,Vision,ResNet-50,100.00%,77.46%,86.89%,0.1364,0.2254,22.54%
5,VLM,CLIP,100.00%,78.53%,87.32%,0.1333,0.2147,21.47%
6,Vision,VGG-16,100.00%,78.97%,87.32%,0.1341,0.2103,21.03%
7,Vision,DenseNet-121,100.00%,79.03%,87.67%,0.1318,0.2097,20.97%
8,VLM,SigLIP,100.00%,79.28%,87.27%,0.1376,0.2072,20.72%
9,Vision,ConvNeXt-Tiny,100.00%,80.23%,89.04%,0.1137,0.1977,19.77%



=== BẢNG TỔNG HỢP - UNTARGETED ATTACK ===


,Category,Source Model,Clean Acc,Adv Acc,Adv NED,Adv CER,Adv WER,ASR
0,Vision,ViT-Base,100.00%,74.64%,84.81%,0.1601,0.2536,25.36%
1,Vision,DenseNet-121,100.00%,76.52%,86.30%,0.1442,0.2348,23.48%
2,VLM,CLIP,100.00%,76.96%,86.41%,0.1409,0.2304,23.04%
3,VLM,EVA-CLIP,100.00%,77.65%,86.25%,0.1465,0.2235,22.35%
4,VLM,OpenCLIP,100.00%,77.90%,87.10%,0.1365,0.2210,22.10%
5,VLM,MetaCLIP,100.00%,78.84%,86.88%,0.1450,0.2116,21.16%
6,Vision,VGG-16,100.00%,79.16%,87.46%,0.1318,0.2084,20.84%
7,VLM,SigLIP,100.00%,79.72%,88.07%,0.1268,0.2028,20.28%
8,Vision,ResNet-50,100.00%,79.79%,88.48%,0.1198,0.2021,20.21%
9,Vision,ConvNeXt-Tiny,100.00%,81.04%,89.38%,0.1107,0.1896,18.96%



=== BẢNG GỘP CẢ 20 PERTURBATION ===


,Category,Source Model,Attack Type,Clean Acc,Adv Acc,Adv NED,Adv CER,Adv WER,ASR
0,Vision,ViT-Base,Targeted,100.00%,76.21%,85.86%,0.1495,0.2379,23.79%
1,VLM,OpenCLIP,Targeted,100.00%,76.77%,86.22%,0.1434,0.2323,23.23%
2,VLM,MetaCLIP,Targeted,100.00%,76.84%,86.06%,0.1469,0.2316,23.16%
3,VLM,EVA-CLIP,Targeted,100.00%,77.21%,86.77%,0.1415,0.2279,22.79%
4,Vision,ResNet-50,Targeted,100.00%,77.46%,86.89%,0.1364,0.2254,22.54%
5,VLM,CLIP,Targeted,100.00%,78.53%,87.32%,0.1333,0.2147,21.47%
6,Vision,VGG-16,Targeted,100.00%,78.97%,87.32%,0.1341,0.2103,21.03%
7,Vision,DenseNet-121,Targeted,100.00%,79.03%,87.67%,0.1318,0.2097,20.97%
8,VLM,SigLIP,Targeted,100.00%,79.28%,87.27%,0.1376,0.2072,20.72%
9,Vision,ConvNeXt-Tiny,Targeted,100.00%,80.23%,89.04%,0.1137,0.1977,19.77%



Đã lưu bảng đầy đủ tại: /kaggle/working/ocr_uap_comparison_20.csv
